In [48]:
# import libraries
import pandas as pd
import numpy as np
import yfinance as yf
import math
from scipy import stats

## Load the Stock List

In [49]:
tickers = pd.read_csv('malaysia_stocks.csv')
tickers.head()

,Company Name,Ticker
0,Hong Leong Bank Berhad,5819.KL
1,Malayan Banking Berhad,1155.KL
2,Hong Leong Financial Group Berhad,1082.KL
3,QL Resources Berhad,7084.KL
4,Maxis Berhad,6012.KL


## Evaluate and Compare Stocks Based on Financial Ratios

In [50]:
def fetch_values_of_stocks(tickers):

    value_cols = [
        "Ticker",
        "Price",
        "PE-Ratio",
        "PB-Ratio",
        "PS-Ratio",
        "EV/EBITDA",
        "EV/GP",
    ]

    value_df = pd.DataFrame(columns=value_cols)
    for ticker in tickers:
        stock = yf.Ticker(ticker)
        price = stock.history(period="1d")['Close'].iloc[-1]

        financials = stock.financials
        balance_sheet = stock.balance_sheet
        info = stock.cashflow

        pe_ratio = stock.info.get("forwardPE", np.nan)
        pb_ratio = stock.info.get("priceToBook", np.nan)
        ps_ratio = stock.info.get("priceToSalesTrailing12Months", np.nan)
        ev = stock.info.get("enterpriseValue", np.nan)
        ebitda = stock.info.get("ebitda", np.nan)
        evEbitda = ev / ebitda if ev and ebitda else np.nan
        grossProfit = stock.info.get("grossMargins" ,np.nan) * stock.info.get("totalRevenue", np.nan)
        evGrossProfit = ev / grossProfit if ev and grossProfit else np.nan

        value_df.loc[len(value_df)] = [
            ticker, 
            price, 
            pe_ratio, 
            pb_ratio, 
            ps_ratio, 
            evEbitda, 
            evGrossProfit
        ]

    return value_df

In [51]:
tickers_list = tickers['Ticker'].values.tolist()

df = fetch_values_of_stocks(tickers_list)
df

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA,EV/GP
0,5819.KL,20.639999,8.782979,1.077076,6.239694,NaN,NaN
1,1155.KL,9.950000,11.436781,1.276459,4.231721,NaN,NaN
2,1082.KL,17.020000,5.386076,0.600522,2.683954,NaN,NaN
3,7084.KL,4.460000,31.857143,5.033860,2.278630,18.160977,9.857547
4,6012.KL,3.660000,19.263159,4.796855,2.726672,10.984090,7.890748
5,1961.KL,3.960000,18.857143,1.997982,2.167383,15.694915,12.182025
6,2445.KL,20.459999,18.106194,1.594700,0.934052,11.197889,10.184033
7,6033.KL,18.940001,19.358137,2.659365,5.804794,11.079301,16.701759
8,5347.KL,13.220000,15.927711,1.264588,1.189922,8.747135,6.560316
9,6947.KL,3.660000,20.333332,2.677396,3.337134,9.931296,6.693761


In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29 entries, 0 to 28
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ticker     29 non-null     object 
 1   Price      29 non-null     float64
 2   PE-Ratio   29 non-null     float64
 3   PB-Ratio   29 non-null     float64
 4   PS-Ratio   29 non-null     float64
 5   EV/EBITDA  23 non-null     float64
 6   EV/GP      23 non-null     float64
dtypes: float64(6), object(1)
memory usage: 1.8+ KB


## Take Care of Null Values

In [53]:
value_cols = [
        "PE-Ratio", 
        "PB-Ratio", 
        "PS-Ratio", 
        "EV/EBITDA", 
        "EV/GP"
    ]

for col in value_cols:
    df[col] = df[col].fillna(df[col].mean())
    
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29 entries, 0 to 28
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ticker     29 non-null     object 
 1   Price      29 non-null     float64
 2   PE-Ratio   29 non-null     float64
 3   PB-Ratio   29 non-null     float64
 4   PS-Ratio   29 non-null     float64
 5   EV/EBITDA  29 non-null     float64
 6   EV/GP      29 non-null     float64
dtypes: float64(6), object(1)
memory usage: 1.8+ KB


## Find the Percentile Scores

In [54]:
percentile_metrics = {
    "PE-Ratio" : "PE-Ratio_Percentile", 
    "PB-Ratio" : "PB-Ratio_Percentile",
    "PS-Ratio" : "PS-Ratio_Percentile",
    "EV/EBITDA" : "EV/EBITDA_Percentile",
    "EV/GP" : "EV/GP_Percentile"
}


for metric, percentile in percentile_metrics.items():
    df[percentile] = df[metric].apply(lambda x: stats.percentileofscore(df[metric], x) / 100)

df.head()

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA_Percentile,EV/GP_Percentile
0,5819.KL,20.639999,8.782979,1.077076,6.239694,14.447330,11.345351,0.103448,0.275862,1.000000,0.637931,0.603448
1,1155.KL,9.950000,11.436781,1.276459,4.231721,14.447330,11.345351,0.310345,0.413793,0.896552,0.637931,0.603448
2,1082.KL,17.020000,5.386076,0.600522,2.683954,14.447330,11.345351,0.034483,0.034483,0.482759,0.637931,0.603448
3,7084.KL,4.460000,31.857143,5.033860,2.278630,18.160977,9.857547,0.862069,0.931034,0.413793,0.862069,0.379310
4,6012.KL,3.660000,19.263159,4.796855,2.726672,10.984090,7.890748,0.620690,0.896552,0.517241,0.413793,0.241379


## Find the Mean

In [55]:
from statistics import mean 

df['Value Score'] = df[[value for value in percentile_metrics.values()]].mean(axis = 1)

df

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA_Percentile,EV/GP_Percentile,Value Score
0,5819.KL,20.639999,8.782979,1.077076,6.239694,14.447330,11.345351,0.103448,0.275862,1.000000,0.637931,0.603448,0.524138
1,1155.KL,9.950000,11.436781,1.276459,4.231721,14.447330,11.345351,0.310345,0.413793,0.896552,0.637931,0.603448,0.572414
2,1082.KL,17.020000,5.386076,0.600522,2.683954,14.447330,11.345351,0.034483,0.034483,0.482759,0.637931,0.603448,0.358621
3,7084.KL,4.460000,31.857143,5.033860,2.278630,18.160977,9.857547,0.862069,0.931034,0.413793,0.862069,0.379310,0.689655
4,6012.KL,3.660000,19.263159,4.796855,2.726672,10.984090,7.890748,0.620690,0.896552,0.517241,0.413793,0.241379,0.537931
5,1961.KL,3.960000,18.857143,1.997982,2.167383,15.694915,12.182025,0.586207,0.586207,0.379310,0.758621,0.724138,0.606897
6,2445.KL,20.459999,18.106194,1.594700,0.934052,11.197889,10.184033,0.551724,0.482759,0.103448,0.482759,0.413793,0.406897
7,6033.KL,18.940001,19.358137,2.659365,5.804794,11.079301,16.701759,0.655172,0.724138,0.965517,0.448276,0.896552,0.737931
8,5347.KL,13.220000,15.927711,1.264588,1.189922,8.747135,6.560316,0.482759,0.379310,0.275862,0.206897,0.103448,0.289655
9,6947.KL,3.660000,20.333332,2.677396,3.337134,9.931296,6.693761,0.689655,0.793103,0.724138,0.310345,0.137931,0.531034


## Sort the Stocks

In [56]:
df = df.sort_values(by="Value Score", ascending=False)
df

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA_Percentile,EV/GP_Percentile,Value Score
19,4707.KL,97.559998,41.766740,35.566895,3.599657,30.892040,12.647595,0.965517,1.000000,0.793103,0.931034,0.793103,0.896552
25,5211.KL,5.780000,32.450035,2.602431,3.667070,46.067167,23.350242,0.931034,0.689655,0.827586,1.000000,1.000000,0.889655
28,3867.KL,30.719999,32.020010,2.945068,2.877006,12.204669,22.726503,0.896552,0.827586,0.655172,0.517241,0.965517,0.772414
7,6033.KL,18.940001,19.358137,2.659365,5.804794,11.079301,16.701759,0.655172,0.724138,0.965517,0.448276,0.896552,0.737931
22,5225.KL,7.990000,88.777770,2.317958,2.832305,17.151281,10.529937,1.000000,0.655172,0.586207,0.793103,0.448276,0.696552
3,7084.KL,4.460000,31.857143,5.033860,2.278630,18.160977,9.857547,0.862069,0.931034,0.413793,0.862069,0.379310,0.689655
16,5296.KL,1.660000,20.750000,7.942584,3.288216,17.162087,7.502053,0.724138,0.965517,0.689655,0.827586,0.206897,0.682759
5,1961.KL,3.960000,18.857143,1.997982,2.167383,15.694915,12.182025,0.586207,0.586207,0.379310,0.758621,0.724138,0.606897
17,1295.KL,4.300000,11.315790,1.419611,5.755365,14.447330,11.345351,0.275862,0.448276,0.931034,0.637931,0.603448,0.579310
1,1155.KL,9.950000,11.436781,1.276459,4.231721,14.447330,11.345351,0.310345,0.413793,0.896552,0.637931,0.603448,0.572414
